# 02 — Graph Construction · Phase 3

**Purpose:** Load the built graph, verify every assertion, visualise the node distribution, edge structure, feature correlations, and geographic split boundaries.

**Pre-condition:** Run `python scripts/phase3_build_graph.py` first. This notebook is for verification and exploration only — it does not rebuild the graph.

**What you confirm here before Phase 4:**
- Graph has the correct number of nodes, features, and edges
- val_mask is not zero (previous project bug)
- No geographic overlap between train/val/test splits
- Target variable y is quantile-transformed (mean≈0, std≈1)
- Feature matrix has no NaN or Inf values
- Node distribution across Greece looks spatially correct

---

In [1]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

import torch
from torch_geometric.data import Data

# Add src/ to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wildfire_gnn.utils import load_yaml_config, set_seed

config  = load_yaml_config(PROJECT_ROOT / 'configs' / 'gnn_config.yaml')
set_seed(config['training']['seed'])

p           = config['paths']
GRAPH_PATH  = PROJECT_ROOT / p['graph_data']
SPLIT_PATH  = PROJECT_ROOT / p['spatial_split_path']
FEAT_PATH   = PROJECT_ROOT / p['feature_names']
FIG_DIR     = PROJECT_ROOT / p['figures_dir']
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Graph path   : {GRAPH_PATH}')
print(f'Graph exists : {GRAPH_PATH.exists()}')

Project root : d:\wildfire\spatiotemporal_wildfire_gnn
Graph path   : d:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt
Graph exists : False


# Load graph and build summary

In [ ]:
if not GRAPH_PATH.exists():
    print('ERROR: graph_data_enriched.pt not found.')
    print('Run: python scripts/phase3_build_graph.py')
    raise FileNotFoundError(str(GRAPH_PATH))

graph = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)

print('Graph loaded successfully')
print()
print(f'  num_nodes          : {graph.num_nodes:,}')
print(f'  num_node_features  : {graph.num_node_features}')
print(f'  num_edges          : {graph.num_edges:,}')
print(f'  avg edges per node : {graph.num_edges / graph.num_nodes:.1f}')
print()
print(f'  x shape            : {tuple(graph.x.shape)}')
print(f'  y shape            : {tuple(graph.y.shape)}')
print(f'  pos shape          : {tuple(graph.pos.shape)}')
print(f'  edge_index shape   : {tuple(graph.edge_index.shape)}')
print()
print(f'  train_mask.sum()   : {graph.train_mask.sum():,}')
print(f'  val_mask.sum()     : {graph.val_mask.sum():,}')
print(f'  test_mask.sum()    : {graph.test_mask.sum():,}')
print()
print(f'  y mean             : {float(graph.y.mean()):.4f}  (should be near 0)')
print(f'  y std              : {float(graph.y.std()):.4f}   (should be near 1)')
print(f'  y_raw min          : {float(graph.y_raw.min()):.6f}')
print(f'  y_raw max          : {float(graph.y_raw.max()):.4f}')

# Critical assertion

In [ ]:
print('Running Phase 3 completion assertions...')
print('=' * 55)

failures = []

def check(condition, message, fix=''):
    sym = '✓' if condition else '✗'
    print(f'  {sym}  {message}')
    if not condition:
        failures.append((message, fix))

# Node count
check(graph.num_nodes > 100_000,
      f'num_nodes={graph.num_nodes:,} > 100k',
      'Re-run phase3_build_graph.py with --stride 6')

check(graph.num_nodes < 600_000,
      f'num_nodes={graph.num_nodes:,} < 600k (memory feasible)',
      'Use --stride 6 or higher')

# Feature count
check(graph.num_node_features in (53, 58),
      f'num_features={graph.num_node_features} is 53 (no DEM) or 58 (with DEM)',
      'Check feature_engineering.py output')

# Edge count
check(graph.num_edges > graph.num_nodes * 2,
      f'num_edges={graph.num_edges:,} > 2×nodes (graph is connected)',
      'Check build_pixel_grid_edges() in graph_builder.py')

# Target transformation
y_mean = float(graph.y.mean())
y_std  = float(graph.y.std())
check(abs(y_mean) < 0.5,
      f'y.mean()={y_mean:.4f} near 0 (QuantileTransformer applied)',
      'Transform was not applied or was applied twice')

check(0.5 < y_std < 2.0,
      f'y.std()={y_std:.4f} near 1 (QuantileTransformer applied)',
      'Check TargetTransformer.transform() in target_engineering.py')

# No NaN in features
has_nan = torch.isnan(graph.x).any()
check(not has_nan,
      'Feature matrix x has no NaN values',
      'Check nan_to_num() calls in feature_engineering.py')

# No Inf in features
has_inf = torch.isinf(graph.x).any()
check(not has_inf,
      'Feature matrix x has no Inf values',
      'Check gradient computation in add_spatial_gradients()')

# Split masks non-zero
check(graph.val_mask.sum() > 0,
      f'val_mask.sum()={graph.val_mask.sum():,} > 0 (not zero placeholder)',
      'CRITICAL: val rows range in config produces no nodes — check split.val_rows')

check(graph.test_mask.sum() > 0,
      f'test_mask.sum()={graph.test_mask.sum():,} > 0',
      'Check split.test_rows range in gnn_config.yaml')

# No geographic overlap
tv_overlap = int((graph.train_mask & graph.val_mask).sum())
tt_overlap = int((graph.train_mask & graph.test_mask).sum())
check(tv_overlap == 0,
      f'Train/Val overlap = {tv_overlap} (must be 0)',
      'CRITICAL geographic leakage — check build_geographic_split()')

check(tt_overlap == 0,
      f'Train/Test overlap = {tt_overlap} (must be 0)',
      'CRITICAL geographic leakage — check build_geographic_split()')

# All nodes covered
covered = int(graph.train_mask.sum() + graph.val_mask.sum() + graph.test_mask.sum())
check(covered == graph.num_nodes,
      f'All {graph.num_nodes:,} nodes covered by exactly one split',
      'Check test_rows upper bound in gnn_config.yaml')

# edge_index valid range
check(int(graph.edge_index.max()) < graph.num_nodes,
      f'edge_index max={int(graph.edge_index.max())} < num_nodes={graph.num_nodes}',
      'Edge index contains out-of-range node index')

# dtype checks
check(graph.x.dtype == torch.float32,
      f'x.dtype = {graph.x.dtype} (float32)',
      'Cast x to float32 in graph assembly')

check(graph.edge_index.dtype == torch.long,
      f'edge_index.dtype = {graph.edge_index.dtype} (torch.long/int64)',
      'Cast edge_index to torch.long')

print('=' * 55)
if failures:
    print(f'\n  FAILED: {len(failures)} assertion(s)')
    for msg, fix in failures:
        print(f'  ✗  {msg}')
        if fix:
            print(f'     Fix: {fix}')
    print('\n  Do NOT proceed to Phase 4 until all assertions pass.')
else:
    print(f'\n  ALL {15} ASSERTIONS PASSED — ready for Phase 4')

# Feature names and group breakdown

In [ ]:
if FEAT_PATH.exists():
    with open(FEAT_PATH) as f:
        feature_names = json.load(f)
    print(f'Total features: {len(feature_names)}')
    print()

    groups = [
        ('Base rasters',      [n for n in feature_names if n in ['CFL','FSP_Index','Ignition_Prob','Struct_Exp_Index']]),
        ('DEM terrain',       [n for n in feature_names if n.startswith('dem_')]),
        ('Fuel one-hot',      [n for n in feature_names if n.startswith('fuel_')]),
        ('Interactions',      [n for n in feature_names if n.startswith('interact_')]),
        ('Multi-scale stats', [n for n in feature_names if '_mean_' in n or '_std_' in n]),
        ('Spatial gradients', [n for n in feature_names if '_grad_' in n]),
        ('Node degree',       [n for n in feature_names if n == 'node_degree']),
    ]

    print(f'  {"Group":<22} {"Count":<8} Features')
    print(f'  {"-"*70}')
    for gname, gfeats in groups:
        feat_str = ', '.join(gfeats[:4])
        if len(gfeats) > 4:
            feat_str += f' ... (+{len(gfeats)-4} more)'
        print(f'  {gname:<22} {len(gfeats):<8} {feat_str}')
    total = sum(len(g) for _, g in groups)
    print(f'  {"-"*70}')
    print(f'  {"TOTAL":<22} {total}')
else:
    print('feature_names.json not found — run phase3_build_graph.py first')

# Spatial distribution of subsampled nodes

In [ ]:
# pos contains (original_row, original_col) for each node
pos     = graph.pos.numpy()
rows_np = pos[:, 0].astype(int)
cols_np = pos[:, 1].astype(int)

# Get split boundaries from config
train_end = config['split']['train_rows'][1]
val_end   = config['split']['val_rows'][1]

# Build 2D density map of subsampled nodes
H, W = 7597, 7555
# Downsample for plotting (every 6th pixel)
node_map = np.zeros((H, W), dtype=np.uint8)
node_map[rows_np, cols_np] = 1

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# Left: node distribution with split lines
ax = axes[0]
ax.imshow(node_map, cmap='Greens', interpolation='nearest', aspect='equal')

ax.axhline(train_end, color='#2980B9', lw=2, linestyle='--',
           label=f'Train/Val split (row {train_end})')
ax.axhline(val_end,   color='#E67E22', lw=2, linestyle='--',
           label=f'Val/Test split  (row {val_end})')

# Shade the three regions
ax.axhspan(0,         train_end, alpha=0.04, color='#2ECC71', label='Train region')
ax.axhspan(train_end, val_end,   alpha=0.10, color='#F39C12', label='Val region')
ax.axhspan(val_end,   H,         alpha=0.04, color='#E74C3C', label='Test region')

ax.set_title(
    f'Subsampled node distribution — {graph.num_nodes:,} nodes\n'
    f'Geographic block split (north→south)',
    fontsize=11
)
ax.set_xlabel('Column (pixel)')
ax.set_ylabel('Row (pixel)')
ax.legend(loc='lower right', fontsize=8)
ax.set_axis_off()

# Right: burn probability spatial map (y_raw)
ax2 = axes[1]
bp_map = np.full((H, W), np.nan)
bp_map[rows_np, cols_np] = graph.y_raw.numpy().ravel()

im = ax2.imshow(bp_map, cmap='YlOrRd', vmin=0, vmax=0.12,
                interpolation='nearest', aspect='equal')
plt.colorbar(im, ax=ax2, label='Burn Probability', fraction=0.03, pad=0.02)

ax2.axhline(train_end, color='#2980B9', lw=1.5, linestyle='--')
ax2.axhline(val_end,   color='#E67E22', lw=1.5, linestyle='--')

ax2.set_title(
    f'Burn Probability (original scale) on subsampled nodes\n'
    f'min={float(graph.y_raw.min()):.5f}  '
    f'max={float(graph.y_raw.max()):.4f}  '
    f'mean={float(graph.y_raw.mean()):.5f}',
    fontsize=11
)
ax2.set_axis_off()

plt.tight_layout()
plt.savefig(FIG_DIR / 'p3_node_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/p3_node_distribution.png')

# Split Statistic table

In [ ]:
splits = {
    'train': graph.train_mask,
    'val':   graph.val_mask,
    'test':  graph.test_mask,
}

print(f'Split statistics  (N = {graph.num_nodes:,} total nodes)')
print('=' * 70)
print(f'  {"Split":<8} {"Nodes":>9} {"Pct":>7}  {"BP mean":>9} {"BP std":>9} '
      f'{"Row min":>9} {"Row max":>9}')
print(f'  {"-"*68}')

for name, mask in splits.items():
    idx      = mask.nonzero(as_tuple=True)[0]
    n        = int(mask.sum())
    pct      = 100 * n / graph.num_nodes
    bp_vals  = graph.y_raw[idx].numpy().ravel()
    row_vals = graph.pos[idx, 0].numpy()
    print(f'  {name:<8} {n:>9,} {pct:>6.1f}%  '
          f'{bp_vals.mean():>9.5f} {bp_vals.std():>9.5f} '
          f'{int(row_vals.min()):>9} {int(row_vals.max()):>9}')

print('=' * 70)
print()
print('Geographic disjointness verification:')
print(f'  train rows: {int(graph.pos[graph.train_mask][:,0].min())} — {int(graph.pos[graph.train_mask][:,0].max())}')
print(f'  val   rows: {int(graph.pos[graph.val_mask][:,0].min())} — {int(graph.pos[graph.val_mask][:,0].max())}')
print(f'  test  rows: {int(graph.pos[graph.test_mask][:,0].min())} — {int(graph.pos[graph.test_mask][:,0].max())}')
print()

# Confirm the row ranges are disjoint
train_max = int(graph.pos[graph.train_mask][:,0].max())
val_min   = int(graph.pos[graph.val_mask][:,0].min())
val_max   = int(graph.pos[graph.val_mask][:,0].max())
test_min  = int(graph.pos[graph.test_mask][:,0].min())

assert train_max < val_min, f'Train bleeds into Val: train_max={train_max}, val_min={val_min}'
assert val_max   < test_min, f'Val bleeds into Test: val_max={val_max}, test_min={test_min}'
print('✓  Row ranges are geographically disjoint — no leakage')

# Feature statistics per group

In [ ]:
if not FEAT_PATH.exists():
    print('feature_names.json not found')
else:
    with open(FEAT_PATH) as f:
        feature_names = json.load(f)

    X = graph.x.numpy()

    print(f'Feature matrix: {X.shape}')
    print(f'NaN count     : {np.isnan(X).sum()}')
    print(f'Inf count     : {np.isinf(X).sum()}')
    print()

    print(f'  {"Feature":<30} {"mean":>10} {"std":>10} {"min":>10} {"max":>10}')
    print(f'  {"-"*72}')

    groups_order = [
        ('BASE', ['CFL','FSP_Index','Ignition_Prob','Struct_Exp_Index']),
        ('DEM',  [n for n in feature_names if n.startswith('dem_')]),
        ('FUEL (first 3)', [n for n in feature_names if n.startswith('fuel_')][:3]),
        ('INTERACT', [n for n in feature_names if n.startswith('interact_')]),
        ('MULTISCALE (sample)', [n for n in feature_names if '_mean_3x3' in n][:3]),
        ('GRADIENT (sample)',   [n for n in feature_names if '_grad_x' in n][:3]),
    ]

    for gname, gfeats in groups_order:
        if not gfeats:
            continue
        print(f'  ── {gname}')
        for fname in gfeats:
            if fname not in feature_names:
                continue
            idx = feature_names.index(fname)
            col = X[:, idx]
            print(f'  {fname:<30} {col.mean():>10.4f} {col.std():>10.4f} '
                  f'{col.min():>10.4f} {col.max():>10.4f}')
    print()

# Feature correlation with target (train split only)

In [ ]:
if not FEAT_PATH.exists():
    print('feature_names.json not found')
else:
    with open(FEAT_PATH) as f:
        feature_names = json.load(f)

    X_train = graph.x[graph.train_mask].numpy()
    y_train = graph.y_raw[graph.train_mask].numpy().ravel()

    # Pearson correlation of each feature with raw burn probability
    correlations = []
    for i, fname in enumerate(feature_names):
        col = X_train[:, i]
        if col.std() < 1e-8:   # skip constant features
            correlations.append(0.0)
            continue
        r = float(np.corrcoef(col, y_train)[0, 1])
        correlations.append(r if not np.isnan(r) else 0.0)

    corr_df = pd.DataFrame({'feature': feature_names, 'pearson_r': correlations})
    corr_df  = corr_df.sort_values('pearson_r', key=abs, ascending=False)

    print('Top 20 features by |Pearson r| with burn probability (train split):')
    print(corr_df.head(20).to_string(index=False))
    print()

    # Plot top 30
    top30  = corr_df.head(30)
    colors = ['#E74C3C' if r > 0 else '#2980B9' for r in top30['pearson_r']]

    fig, ax = plt.subplots(figsize=(8, 10))
    bars = ax.barh(range(len(top30)), top30['pearson_r'].values, color=colors, height=0.7)
    ax.set_yticks(range(len(top30)))
    ax.set_yticklabels(top30['feature'].values, fontsize=9)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Pearson r with burn probability')
    ax.set_title('Top 30 features by correlation with Burn_Prob\n(train split only — no leakage)', fontsize=11)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'p3_feature_correlations.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved: reports/figures/p3_feature_correlations.png')

# Target distribution: transformed vs raw

In [ ]:
y_raw  = graph.y_raw.numpy().ravel()
y_t    = graph.y.numpy().ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw BP distribution
ax = axes[0]
ax.hist(y_raw, bins=80, color='#C0392B', alpha=0.85, edgecolor='none')
ax.axvline(float(np.mean(y_raw)), color='navy', lw=2, linestyle='--',
           label=f'mean={float(np.mean(y_raw)):.4f}')
ax.set_xlabel('Burn Probability (original scale)')
ax.set_ylabel('Node count')
ax.set_title('y_raw — raw burn probability\n(from graph.y_raw, subsampled nodes)', fontsize=10)
ax.legend(fontsize=9)

# Transformed distribution
ax2 = axes[1]
ax2.hist(y_t, bins=80, color='#1E8449', alpha=0.85, edgecolor='none')
ax2.axvline(float(np.mean(y_t)), color='navy', lw=2, linestyle='--',
            label=f'mean={float(np.mean(y_t)):.4f}')
ax2.set_xlabel('Transformed Burn Probability')
ax2.set_ylabel('Node count')
ax2.set_title('y — quantile-transformed (graph.y)\nNear-Gaussian: mean≈0, std≈1', fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'p3_target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/p3_target_distributions.png')
print()

# Critical: confirm transform not applied twice
assert abs(float(np.mean(y_t))) < 0.5, \
    f'y mean={np.mean(y_t):.4f} — transform may have been applied twice!'
assert 0.5 < float(np.std(y_t)) < 2.0, \
    f'y std={np.std(y_t):.4f} — transform looks wrong'
print('✓  Target transform validated — safe for Gaussian NLL loss')

# Edge degree distribution

In [ ]:
from torch_geometric.utils import degree

deg = degree(graph.edge_index[0], num_nodes=graph.num_nodes).numpy()

print('Edge degree statistics:')
print(f'  min degree  : {int(deg.min())}  (boundary/island nodes)')
print(f'  max degree  : {int(deg.max())}  (should be 8 for interior nodes)')
print(f'  mean degree : {deg.mean():.2f}')
print(f'  std degree  : {deg.std():.2f}')
print()

unique, counts = np.unique(deg.astype(int), return_counts=True)
print('Degree distribution:')
for d, c in zip(unique, counts):
    pct = 100 * c / len(deg)
    bar = '█' * int(pct / 2)
    label = 'interior' if d == 8 else 'boundary/coast'
    print(f'  degree {d}: {c:>8,} nodes ({pct:5.1f}%)  {bar}  {label}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(unique, counts, color='#2C3E50', alpha=0.85, width=0.7)
ax.set_xlabel('Node degree (number of spatial neighbors)')
ax.set_ylabel('Number of nodes')
ax.set_xticks(list(range(1, 9)))
ax.set_title('8-connected pixel grid edge degree distribution\n'
             'Degree=8: interior  |  Degree<8: coastline or dataset boundary',
             fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p3_degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/p3_degree_distribution.png')

# Memory footprint estimate for GPU training

In [ ]:
N = graph.num_nodes
F = graph.num_node_features
E = graph.num_edges
H = config['model']['hidden_channels']
heads = config['model']['heads']

# Memory estimate (bytes)
x_bytes          = N * F * 4          # float32
edge_bytes       = E * 2 * 8          # int64 pairs
y_bytes          = N * 1 * 4          # float32
hidden_bytes     = N * H * 4          # one hidden layer
attention_bytes  = E * heads * 4      # attention coefficients

total_data   = x_bytes + edge_bytes + y_bytes
total_model  = hidden_bytes + attention_bytes
total_est    = total_data + total_model * config['model']['num_layers']

def mb(b): return b / 1024**2

print('GPU memory estimate for full-graph training:')
print(f'  Node feature matrix x  : {mb(x_bytes):.1f} MB')
print(f'  Edge index             : {mb(edge_bytes):.1f} MB')
print(f'  Target y               : {mb(y_bytes):.1f} MB')
print(f'  Hidden states (1 layer): {mb(hidden_bytes):.1f} MB')
print(f'  Attention coeffs (8h)  : {mb(attention_bytes):.1f} MB')
print(f'  ─────────────────────────────────────────')
print(f'  Estimated minimum VRAM : {mb(total_est):.1f} MB  ({mb(total_est)/1024:.1f} GB)')
print(f'  Add 2–3× for gradients + activations')
print(f'  Recommended VRAM       : {mb(total_est*2.5)/1024:.0f}–{mb(total_est*3)/1024:.0f} GB')
print()

if mb(total_est*2.5) > 16*1024:
    print('⚠  Estimated usage exceeds 16 GB — consider reducing --stride')
    print('   or enabling mini-batch with config training.batch_size=2048')
else:
    print('✓  Estimated usage fits within 16 GB VRAM — full-graph training feasible')

# Phase 3 completion checklist

In [ ]:
print('=' * 55)
print('  PHASE 3 COMPLETION CHECKLIST')
print('=' * 55)

items = [
    ('graph_data_enriched.pt saved',   GRAPH_PATH.exists()),
    ('splits_enriched.npz saved',      SPLIT_PATH.exists()),
    ('feature_names.json saved',       FEAT_PATH.exists()),
    ('num_nodes > 100k',               graph.num_nodes > 100_000),
    ('num_features 53 or 58',          graph.num_node_features in (53, 58)),
    ('y.mean() near 0',                abs(float(graph.y.mean())) < 0.5),
    ('y.std() near 1',                 0.5 < float(graph.y.std()) < 2.0),
    ('val_mask.sum() > 0',             int(graph.val_mask.sum()) > 0),
    ('No train/val overlap',           int((graph.train_mask & graph.val_mask).sum()) == 0),
    ('No train/test overlap',          int((graph.train_mask & graph.test_mask).sum()) == 0),
    ('All nodes covered',              int(graph.train_mask.sum()+graph.val_mask.sum()+graph.test_mask.sum()) == graph.num_nodes),
    ('No NaN in x',                    not torch.isnan(graph.x).any().item()),
    ('No Inf in x',                    not torch.isinf(graph.x).any().item()),
    ('edge_index dtype = int64',       graph.edge_index.dtype == torch.long),
    ('x dtype = float32',              graph.x.dtype == torch.float32),
]

all_ok = True
for label, ok in items:
    sym = '✓' if ok else '✗'
    print(f'  {sym}  {label}')
    all_ok = all_ok and ok

print('=' * 55)
if all_ok:
    print()
    print('  ALL CHECKS PASSED')
    print('  Ready for Phase 4: Baseline Models')
    print()
    print('  Load graph in Phase 4 notebook with:')
    print(f"  graph = torch.load('{GRAPH_PATH}', weights_only=False)")
    print(f'  assert graph.num_nodes > 100_000')
    print(f'  assert abs(float(graph.y.mean())) < 0.5')
else:
    print()
    print('  SOME CHECKS FAILED — do not proceed to Phase 4')
    print('  Fix failures above and re-run phase3_build_graph.py')